# Critique Experiment 1 (Strict Pipeline): Shock Train

Methodology aligned with notebook 116:
- strict artifact preparation (author-like postprocess/IR),
- script-based figure build,
- notebook only orchestrates and displays outputs.

In [ ]:
import os
import sys
import shlex
import pathlib
import subprocess

import pandas as pd
import torch
from IPython.display import Image, Markdown, display


def _find_project_root() -> pathlib.Path:
    here = pathlib.Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "src").is_dir() and (p / "scripts" / "build_critique_figures_author_strict.py").is_file():
            return p
    cand = pathlib.Path("/content/econml")
    if (cand / "src").is_dir() and (cand / "scripts" / "build_critique_figures_author_strict.py").is_file():
        return cand
    raise RuntimeError("Project root not found.")


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_ROOT = str(PROJECT_ROOT / "artifacts")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float64"
USE_SELECTED = True
CONS_MODE = "paper"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)


In [ ]:
FIG_DIR = str(PROJECT_ROOT / "artifacts" / "critique_figures_nb120_author_strict")
os.makedirs(FIG_DIR, exist_ok=True)

cmd = [
    sys.executable,
    "scripts/build_critique_figures_author_strict.py",
    "--artifacts-root", ARTIFACTS_ROOT,
    "--fig-dir", FIG_DIR,
    "--device", DEVICE,
    "--dtype", DTYPE,
    "--use-selected", str(USE_SELECTED).lower(),
    "--cons-mode", CONS_MODE,
    "--ensure-postprocess", "true",
    "--ensure-ir", "true",
    "--scenarios", "shock_train",
]
print("Running:", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=True)
print("Done. FIG_DIR=", FIG_DIR)


In [ ]:
for pol in ["taylor", "discretion", "commitment"]:
    p = pathlib.Path(FIG_DIR) / f"critique_shock_train_{pol}.png"
    if p.exists():
        display(Markdown(f"### Shock Train: {pol}"))
        display(Image(filename=str(p), width=980))
    else:
        print("Missing figure:", p)

p_sum = pathlib.Path(FIG_DIR) / "critique_shock_train_summary.csv"
p_del = pathlib.Path(FIG_DIR) / "critique_shock_train_delta.csv"
if p_sum.exists():
    display(Markdown("### Summary"))
    display(pd.read_csv(p_sum))
else:
    print("Missing:", p_sum)
if p_del.exists():
    display(Markdown("### Delta"))
    display(pd.read_csv(p_del))
else:
    print("Missing:", p_del)
